# 07 — Microsoft Fabric Model Registration

## Objective

This notebook registers the frozen Missed Delivery early-warning model in Microsoft Fabric using MLflow.

The objective is not to retrain or optimize the model. The scientific configuration was previously frozen after temporal validation and evaluation.

The registered inference chain is:

**Preprocessing → XGBoost → Sigmoid calibration → Operational threshold → Alert flag**

The frozen configuration is:

- Model: XGBoost Full
- Calibration: Sigmoid
- Operational threshold: 0.08
- Input features: 154
- Model version: 1.0.0
- Threshold status: provisional pending business approval

This notebook performs the following tasks:

1. Validate the Fabric Python environment.
2. Load the frozen model artifacts.
3. Validate the feature contract.
4. Define the MLflow input and output signatures.
5. Package the complete inference chain as an executable MLflow model.
6. Log scientific metrics, parameters, and governance metadata.
7. Register the model as a Microsoft Fabric ML Model.


In [ ]:
import notebookutils

print(notebookutils.nbResPath)

In [ ]:
from pathlib import Path

artifact_dir = Path("builtin/model_artifacts")

print("Exists:", artifact_dir.exists())

for f in artifact_dir.iterdir():
    print(f.name)

In [ ]:
import notebookutils
from pathlib import Path

artifact_dir = Path(notebookutils.nbResPath) / "model_artifacts"
wheel_path = artifact_dir / "missed_delivery-0.1.0-py3-none-any.whl"

print(wheel_path)

In [ ]:
%pip install "builtin/model_artifacts/missed_delivery-0.1.0-py3-none-any.whl"

## 1. Environment Validation

Before loading the serialized model artifacts, the Fabric runtime is validated against the library versions used when the model was frozen.

This step is important because the model pipeline and calibrator were serialized with `joblib`, and loading scikit-learn objects under different library versions may produce compatibility warnings or inconsistent predictions.

The expected environment is:

- Python: 3.12
- pandas: 3.0.5
- NumPy: 2.5.1
- scikit-learn: 1.9.0
- XGBoost: 3.4.1
- joblib: 1.5.3
- PyYAML: 6.0.3

The custom `missed-delivery` wheel was installed at the beginning of the notebook to reproduce the frozen local environment as closely as possible inside Microsoft Fabric.

In [ ]:
import sys
import pandas as pd
import numpy as np
import sklearn
import xgboost
import joblib
import yaml

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("joblib:", joblib.__version__)
print("PyYAML:", yaml.__version__)

In [ ]:
from missed_delivery.inference import predict_batch

print("missed_delivery package: OK")

## 2. Loading the Frozen Model Artifacts

The trained model is not retrained in Microsoft Fabric.

Instead, the previously frozen artifacts are loaded directly from the notebook's built-in resources. This preserves the exact scientific configuration selected during the modeling and evaluation stages.

Two serialized objects are required:

- `md_xgboost_pipeline.joblib`: preprocessing pipeline and the final XGBoost classifier.
- `calibrator.joblib`: sigmoid calibration model implemented as a logistic regression.

Keeping these components separate reflects the original inference architecture:

**Input features → preprocessing → XGBoost raw probability → sigmoid calibration → calibrated probability**

In [ ]:
import joblib
from pathlib import Path

artifact_dir = Path("builtin/model_artifacts")

pipeline = joblib.load(
    artifact_dir / "md_xgboost_pipeline.joblib"
)

calibrator = joblib.load(
    artifact_dir / "calibrator.joblib"
)

print("Pipeline:", type(pipeline))
print("Calibrator:", type(calibrator))

## 3. Loading Model Metadata and Feature Contract

In addition to the serialized model components, the deployment package includes metadata and configuration files that describe the frozen model contract.

These files provide:

- model identity and version information,
- training, calibration, threshold-selection and holdout periods,
- the expected 154 input features and their data types,
- the selected operational threshold and its governance status.

This information is loaded independently from the model artifacts so that the registered Fabric model preserves both the executable inference logic and the scientific configuration used during evaluation.

In [ ]:
import json
import yaml

with open(
    artifact_dir / "model_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    metadata = json.load(f)

with open(
    artifact_dir / "feature_schema.json",
    "r",
    encoding="utf-8"
) as f:
    feature_schema = json.load(f)

with open(
    artifact_dir / "thresholds.yaml",
    "r",
    encoding="utf-8"
) as f:
    thresholds = yaml.safe_load(f)

print("Model:", metadata["model_name"])
print("Version:", metadata["model_version"])
print("Features:", feature_schema["feature_count"])
print("Calibration:", metadata["calibration_method"])
print("Threshold:", metadata["operational_threshold"])
print("Threshold status:", metadata["threshold_status"])

## 4. Building a Synthetic Input Example

MLflow can store an explicit input schema for the registered model.

To define this schema without using confidential production records, a synthetic one-row input is generated from the frozen feature contract.

The example:

- contains exactly the 154 expected input features,
- preserves the feature order,
- respects the categorical and numerical feature types,
- does not contain real business data.

This synthetic input will later be used to infer the MLflow model signature.

In [ ]:
import pandas as pd

expected_features = feature_schema["expected_features"]
categorical_features = set(feature_schema["categorical_features"])

row = {}

for feature in expected_features:
    if feature in categorical_features:
        row[feature] = "UNKNOWN"
    else:
        row[feature] = 0.0

input_example = pd.DataFrame(
    [row],
    columns=expected_features
)

print("Shape:", input_example.shape)
print("Columns:", len(input_example.columns))
print("Categorical features:", len(categorical_features))
print("Numerical features:", len(expected_features) - len(categorical_features))
print("\nDtype distribution:")
print(input_example.dtypes.value_counts())

## 5. Validating the Input Contract and Generating an Output Example

Before creating the MLflow model signature, the synthetic input is passed through the frozen inference chain.

This validation confirms that:

- the 154-feature contract is accepted by the preprocessing pipeline,
- the XGBoost classifier produces a raw probability,
- the sigmoid calibrator transforms that score into a calibrated probability,
- the operational threshold generates the final alert flag.

The resulting output example will be used together with the synthetic input to define the MLflow input and output schema.

In [ ]:
import numpy as np
import pandas as pd

raw_score = pipeline.predict_proba(
    input_example
)[:, 1]

def score_to_logit(score):
    clipped_score = np.clip(
        np.asarray(score, dtype=float),
        1e-6,
        1 - 1e-6
    )
    return np.log(
        clipped_score / (1 - clipped_score)
    ).reshape(-1, 1)


score_logit = score_to_logit(raw_score)

calibrated_probability = calibrator.predict_proba(
    score_logit
)[:, 1]

threshold = float(
    metadata["operational_threshold"]
)

alert_flag = (
    calibrated_probability >= threshold
).astype(int)

output_example = pd.DataFrame({
    "raw_score": raw_score,
    "calibrated_probability": calibrated_probability,
    "threshold": threshold,
    "alert_flag": alert_flag
})

print(output_example)

print("\nOutput shape:", output_example.shape)

print("\nOutput dtypes:")
print(output_example.dtypes)

## 6. Creating the MLflow Model Signature

The MLflow model signature formally defines the expected model inputs and outputs.

The input schema is inferred from the synthetic 154-feature example, while the output schema is inferred from the complete inference result:

- raw XGBoost score,
- calibrated probability,
- operational threshold,
- binary alert flag.

Including the signature improves model governance and makes the registered Fabric ML Model self-describing.

In [ ]:
from mlflow.models import infer_signature

signature = infer_signature(
    input_example,
    output_example
)

print(signature)

In [ ]:
print("Input schema:")
print(signature.inputs)

print("\nOutput schema:")
print(signature.outputs)

In [ ]:
import mlflow
import numpy as np
import pandas as pd
import joblib

class MissedDeliveryModel(mlflow.pyfunc.PythonModel):

    def load_context(self, context):

        self.pipeline = joblib.load(
            context.artifacts["pipeline"]
        )

        self.calibrator = joblib.load(
            context.artifacts["calibrator"]
        )

        self.threshold = 0.08

    def predict(self, context, model_input):

        raw_score = self.pipeline.predict_proba(
            model_input
        )[:, 1]

        clipped_score = np.clip(
            np.asarray(raw_score, dtype=float),
            1e-6,
            1 - 1e-6
        )

        score_logit = np.log(
            clipped_score / (1 - clipped_score)
        ).reshape(-1, 1)

        calibrated_probability = self.calibrator.predict_proba(
            score_logit
        )[:, 1]

        alert_flag = (
            calibrated_probability >= self.threshold
        ).astype(int)

        return pd.DataFrame({
            "raw_score": raw_score,
            "calibrated_probability": calibrated_probability,
            "threshold": self.threshold,
            "alert_flag": alert_flag
        })

print("MissedDeliveryModel wrapper ready")

## 8. Creating the Final MLflow Run

The final MLflow run stores the complete executable model together with its scientific and operational metadata.

The run logs:

- frozen model parameters,
- final temporal holdout metrics,
- threshold-selection metrics,
- governance information,
- the executable inference wrapper,
- the serialized XGBoost pipeline and sigmoid calibrator,
- the 154-feature input example,
- the MLflow input/output signature.

This run is the one that will be promoted to a Microsoft Fabric ML Model.

No model retraining, recalibration, or threshold selection is performed.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import numpy as np

from missed_delivery.inference import predict_batch

artifact_dir = Path("builtin/model_artifacts")

context = SimpleNamespace(
    artifacts={
        "pipeline": str(
            artifact_dir / "md_xgboost_pipeline.joblib"
        ),
        "calibrator": str(
            artifact_dir / "calibrator.joblib"
        ),
    }
)

wrapper = MissedDeliveryModel()
wrapper.load_context(context)

canonical_output = predict_batch(
    input_example,
    artifact_dir=artifact_dir,
    threshold_config_path=(
        artifact_dir / "thresholds.yaml"
    ),
)[[
    "raw_score",
    "calibrated_probability",
    "threshold",
    "alert_flag",
]]

wrapper_output = wrapper.predict(
    None,
    input_example
)

np.testing.assert_allclose(
    wrapper_output["raw_score"],
    canonical_output["raw_score"],
    rtol=0,
    atol=1e-12,
)

np.testing.assert_allclose(
    wrapper_output["calibrated_probability"],
    canonical_output["calibrated_probability"],
    rtol=0,
    atol=1e-12,
)

np.testing.assert_allclose(
    wrapper_output["threshold"],
    canonical_output["threshold"],
    rtol=0,
    atol=0,
)

np.testing.assert_array_equal(
    wrapper_output["alert_flag"],
    canonical_output["alert_flag"],
)

print("Notebook 07 wrapper parity: PASS")
display(wrapper_output)

In [ ]:
import mlflow
from pathlib import Path

artifact_dir = Path("builtin/model_artifacts")

mlflow.set_experiment("exp_md_early_warning")

with mlflow.start_run(
    run_name="md_xgboost_v2_calibration_fix"
) as run:

    mlflow.log_params({
        "model_version": "2.0.0",
        "feature_count": 154,
        "calibration_method": "sigmoid",
        "operational_threshold": 0.08,
        "change_reason": (
            "correct_logit_input_to_frozen_sigmoid_calibrator"
        ),
    })

    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=MissedDeliveryModel(),
        artifacts={
            "pipeline": str(
                artifact_dir / "md_xgboost_pipeline.joblib"
            ),
            "calibrator": str(
                artifact_dir / "calibrator.joblib"
            ),
        },
        input_example=input_example,
        signature=signature,
    )

    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)

logged_model = mlflow.pyfunc.load_model(
    model_info.model_uri
)

print("Logged model loaded:", model_info.model_uri)


In [ ]:
logged_input = input_example.copy()

for col in [
    "Grupo_raiz",
    "customer",
    "uat",
    "destination",
]:
    logged_input[col] = logged_input[col].astype(object)

print(logged_input.dtypes.value_counts())

In [ ]:
logged_output = logged_model.predict(
    logged_input
)

np.testing.assert_allclose(
    logged_output["raw_score"],
    wrapper_output["raw_score"],
    rtol=0,
    atol=1e-12,
)

np.testing.assert_allclose(
    logged_output["calibrated_probability"],
    wrapper_output["calibrated_probability"],
    rtol=0,
    atol=1e-12,
)

np.testing.assert_allclose(
    logged_output["threshold"],
    wrapper_output["threshold"],
    rtol=0,
    atol=0,
)

np.testing.assert_array_equal(
    logged_output["alert_flag"],
    wrapper_output["alert_flag"],
)

print("Logged MLflow model parity: PASS")
display(logged_output)

In [ ]:
import mlflow

model_name = "md_early_warning"

model_uri = model_info.model_uri

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print("Model name:", registered_model.name)
print("Version:", registered_model.version)
print("Source:", registered_model.source)
print("Status:", registered_model.status)

In [ ]:
import mlflow
import numpy as np

registered_model_uri = "models:/md_early_warning/2"

registered_model = mlflow.pyfunc.load_model(
    registered_model_uri
)

registered_output = registered_model.predict(
    logged_input
)

np.testing.assert_allclose(
    registered_output["raw_score"],
    wrapper_output["raw_score"],
    rtol=0,
    atol=1e-12,
)

np.testing.assert_allclose(
    registered_output["calibrated_probability"],
    wrapper_output["calibrated_probability"],
    rtol=0,
    atol=1e-12,
)

np.testing.assert_allclose(
    registered_output["threshold"],
    wrapper_output["threshold"],
    rtol=0,
    atol=0,
)

np.testing.assert_array_equal(
    registered_output["alert_flag"],
    wrapper_output["alert_flag"],
)

print("Registered model V2 parity: PASS")
display(registered_output)